# PlantoAI 200-Species Scalable Training Pipeline
This notebook automatically downloads the base medicinal datasets, constructs the EfficientNetV2 neural architecture, and initiates robust GPU training with auto-checkpoints.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
import urllib.request
import tarfile

# 1. AUTO-DOWNLOADER
DATA_DIR = "./datasets"
os.makedirs(DATA_DIR, exist_ok=True)
url = "https://sourceforge.net/projects/flavia/files/Leaf%20Image%20Dataset/1.0/Leaves.tar.bz2/download"
dest = os.path.join(DATA_DIR, "flavia.tar.bz2")

if not os.path.exists(dest):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, dest)
    with tarfile.open(dest, 'r:bz2') as t:
        t.extractall(DATA_DIR)
    print("✅ Dataset ready.")

# 2. NEURAL ARCHITECTURE (200 SPECIES)
TARGET_CLASSES = 200
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Engine attached to: {device}")

model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
for param in model.parameters(): param.requires_grad = False
model.classifier[1] = nn.Linear(model.classifier[1].in_features, TARGET_CLASSES)
model = model.to(device)

# 3. CHECKPOINT RESILIENCE
CHECKPOINT_PATH = "/kaggle/working/plantoai_checkpoint.pth"
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
start_epoch = 0

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"🔄 Resuming from Epoch {start_epoch}")

print("✅ Pipeline Initialized and Ready for Training!")